# V1: Preprocessing the Dataset

## 1.1 Loading the Dataset

- Load the RunBugRun dataset from Hugging Face.
- Keep only the columns needed for the V1 baseline.
- Preserve columns such as `buggy_code` and `fixed_code` for error analysis.
- Architecture Decision: Decided to remove certain classes that did not have enough examples (< 200) after the single label filtering.

In [ ]:
from src.preprocessing import generate_diff
from datasets import load_dataset
from pathlib import Path
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("iberu/RunBugRun")

buggy_code = []
fixed_code = []
diff_generated = []
labels_generated = []
full_label = []

excluded_v1_classes = ['literal', 'function', 'variable_access', 'io', 'try_catch']

for row in ds['train']: #cleans the data by filering on empty labels, more than one row, taking only the top level label and not in the exclused class
    if not row['labels']:
        continue

    if len(row['labels']) != 1:
        continue

    top_level_label = row['labels'][0].split('.')[0]

    if top_level_label in excluded_v1_classes:
        continue
    buggy_code.append(row['buggy_code'])
    fixed_code.append(row['fixed_code'])
    diff_generated.append(generate_diff(row['buggy_code'], row['fixed_code']))
    labels_generated.append(top_level_label)
    full_label.append(row['labels'][0])

cleaned_df = pd.DataFrame({
    'buggy_code': buggy_code,
    'fixed_code': fixed_code,
    'diff': diff_generated,
    'top_level_label': labels_generated,
    'full_label': full_label
})
Path("data/processed").mkdir(parents=True, exist_ok=True)
cleaned_df.to_csv("data/processed/cleaned_v1.csv", index=False)


C:\Users\Home\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.2 Checking the Dataset (Hugging Face)

- Confirm the number of examples and target labels match.
- Check how many examples remain after each filtering step.
- Check the distribution of the labels

In [14]:
from collections import Counter
print(f'Amount of diff_generated: {len(diff_generated)}')
print(f'Amount of labels_generated: {len(labels_generated)}\n')
print('One example from diff:')
for i in range(1):
    print(diff_generated[i])

print('One example of a label:')
for i in range(1):
    print(labels_generated[i])
    
print(f"\nTotal training examples: {len(ds['train'])}\n")
print(f'Usable examples: {len(diff_generated)}\n')
print(f"Filtered out: {len(ds['train']) - len(diff_generated)}\n")

label_counts = Counter(labels_generated)
print(label_counts) #distribution of the labels

Amount of diff_generated: 35641
Amount of labels_generated: 35641

One example from diff:
-         print(i,"x",j,"=",i*j)
+         print(i,"x",j,"=",i*j,sep="")
One example of a label:
call

Total training examples: 133705

Usable examples: 35641

Filtered out: 98064

Counter({'call': 17165, 'expression': 7415, 'control_flow': 5394, 'assignment': 4586, 'identifier': 1081})


## 1.3 Conclusion:

The RunBugRun training split was reduced from 133,705 examples to 35,641 usable single label examples for the V1 baseline. Rows without labels, rows with multiple labels, and excluded V1 classes were removed.

For each example, a unified diff was generated from the buggy and the fixed code. The resulting dataset contains the original code, generated diff, top level label, and the full label. The data was saved to `data/processed/cleaned.csv`.

The remaining classes are imbalanced, with `call` being the largest class and the `identifier` being the smallest. This was considered during the data splitting and model evaluation.